# Lecture 7 — Class Exercise
## Part-to-Whole: Hierarchical Visualization

> **Push to:** `week07/lecture07_exercise.ipynb`


In [1]:
import pandas as pd
import plotly.express as px
import numpy as np

# Dataset: Global Energy Mix by Country and Source
df = pd.read_csv('../data/global_energy_mix.csv')

# Source type mapping — reuse from lecture
source_category = {
    'Coal': 'Fossil', 'Oil': 'Fossil', 'Natural Gas': 'Fossil',
    'Nuclear': 'Low-carbon', 'Hydro': 'Low-carbon',
    'Wind': 'Renewable', 'Solar': 'Renewable', 'Other Renewables': 'Renewable'
}
df['Source_Type'] = df['Source'].map(source_category)

print(f"Loaded: {len(df)} rows")
print(df.head(10))


Loaded: 103 rows
         Country         Region            Source  Share_pct     TWh  \
0  United States  North America              Coal         10  1015.0   
1  United States  North America               Oil         35  3220.0   
2  United States  North America       Natural Gas         34  3083.0   
3  United States  North America           Nuclear          9   798.0   
4  United States  North America             Hydro          3   339.0   
5  United States  North America              Wind          4   413.0   
6  United States  North America             Solar          3   325.0   
7  United States  North America  Other Renewables          2   229.0   
8          China           Asia              Coal         58  6929.1   
9          China           Asia               Oil         18  1620.0   

  Source_Type  
0      Fossil  
1      Fossil  
2      Fossil  
3  Low-carbon  
4  Low-carbon  
5   Renewable  
6   Renewable  
7   Renewable  
8      Fossil  
9      Fossil  


## Task 1 — Treemap: fossil fuel dependency by country

**What to build:** A treemap showing **fossil fuel TWh only**, broken down by Region → Country → Source (Coal / Oil / Natural Gas).

**Requirements:**
- Filter to fossil sources only before plotting
- Use `path=['Region', 'Country', 'Source']` for the hierarchy
- Colour encodes the fossil source type (Coal / Oil / Natural Gas) with a CVD-safe palette
- Show TWh values in labels — no percentages
- Grey out parent nodes (Region and Country level)
- Insight title naming which region or country is most fossil-dependent

> 💡 `df.loc[df['Source_Type'] == 'Fossil']`


In [5]:
# Task 1
# Okabe-Ito colors distinguish fossil sources while remaining CVD-safe.
fossil_colors = {
    'Coal': '#0072B2',
    'Oil': '#E69F00',
    'Natural Gas': '#009E73',
}

df_fossil = df.loc[df['Source_Type'].eq('Fossil')].copy()

fig_fossil = px.treemap(
    df_fossil,
    path=['Region', 'Country', 'Source'],
    values='TWh',
    color='Source',
    color_discrete_map=fossil_colors,
    labels={'TWh': 'Energy (TWh)', 'Source': 'Fossil source'},
    title='Asia has the largest fossil energy total (35,045 TWh)',
)
fig_fossil.update_traces(
    texttemplate='%{label}<br>%{value:,.1f} TWh',
    hovertemplate='<b>%{label}</b><br>%{value:,.1f} TWh<extra></extra>',
    root_color='white',
)
# Keep source leaves colored and make the aggregated region/country nodes neutral.
fig_fossil.data[0].marker.colors = [
    color if color in fossil_colors.values() else '#D9D9D9'
    for color in fig_fossil.data[0].marker.colors
]
fig_fossil.update_layout(
    template='plotly_white',
    height=650,
    margin=dict(t=80, l=10, r=10, b=10),
)
fig_fossil.show()


## Task 2 — Treemap vs bar: low-carbon energy by country

**What to build:** Build **both** a treemap and a horizontal bar chart showing total low-carbon TWh (Nuclear + Hydro) per country. Then answer the question in a markdown cell below.

**Requirements:**
- Filter to `Source_Type == 'Low-carbon'` and aggregate TWh by country
- Treemap: single-level `path=['All', 'Country']` with a dummy root node labelled `'Low-carbon'`
- Bar chart: sorted by TWh, horizontal orientation, CVD-safe colour
- Both charts show TWh values, not percentages
- Insight title on the bar chart naming the leading country

In [6]:
# Task 2 — charts
df_low_carbon = (
    df.loc[df['Source_Type'].eq('Low-carbon')]
    .groupby('Country', as_index=False)['TWh']
    .sum()
    .sort_values('TWh', ascending=False)
)
df_low_carbon['All'] = 'Low-carbon'
leading_country = df_low_carbon.iloc[0]

fig_low_carbon_treemap = px.treemap(
    df_low_carbon,
    path=['All', 'Country'],
    values='TWh',
    color='All',
    color_discrete_map={'Low-carbon': '#0072B2'},
    labels={'TWh': 'Energy (TWh)'},
    title='Low-carbon energy by country',
)
fig_low_carbon_treemap.update_traces(
    texttemplate='%{label}<br>%{value:,.0f} TWh',
    hovertemplate='<b>%{label}</b><br>%{value:,.0f} TWh<extra></extra>',
)
fig_low_carbon_treemap.update_layout(
    template='plotly_white',
    height=550,
    margin=dict(t=70, l=10, r=10, b=10),
)

fig_low_carbon_bar = px.bar(
    df_low_carbon,
    x='TWh',
    y='Country',
    orientation='h',
    text='TWh',
    color_discrete_sequence=['#0072B2'],
    labels={'TWh': 'Energy (TWh)', 'Country': ''},
    title=(f"{leading_country['Country']} leads low-carbon energy "
           f"({leading_country['TWh']:,.0f} TWh)"),
)
fig_low_carbon_bar.update_traces(
    texttemplate='%{x:,.0f} TWh',
    textposition='outside',
)
fig_low_carbon_bar.update_yaxes(
    categoryorder='array',
    categoryarray=df_low_carbon.sort_values('TWh')['Country'].tolist(),
)
fig_low_carbon_bar.update_xaxes(
    title='Energy (TWh)',
    range=[0, df_low_carbon['TWh'].max() * 1.18],
)
fig_low_carbon_bar.update_layout(
    template='plotly_white',
    height=550,
    margin=dict(t=80, l=10, r=35, b=10),
)

fig_low_carbon_treemap.show()
fig_low_carbon_bar.show()

### Task 2 — reflection

**Which chart tells the ranking story more clearly, and why?**

The horizontal bar chart communicates the ranking more clearly because all countries share a common length scale and the sorted bars make adjacent comparisons immediate. The treemap is better for showing relative part-to-whole contribution, but comparing similarly sized rectangles is harder.

## Task 3 — 100% stacked bar: which region relies most on coal?

**What to build:** A horizontal 100% stacked bar chart showing, for each **Region**, how its fossil energy splits between Coal, Oil and Natural Gas. This is the **same data as your Task 1 treemap**, visualised a second way.

**Requirements:**
- Reuse `df_fossil` from Task 1; aggregate TWh by `Region` × `Source` and convert to **shares of each region's fossil total**
- Horizontal bars, regions **sorted by coal share** (highest at the top)
- Coal is the segment on the baseline (first in `category_orders`) and the **only coloured segment** — keep it the same colour as in Task 1; Oil and Natural Gas in two greys
- Percentage labels on the coal segments only
- Insight title naming the most coal-dependent region and its coal share

> 💡 `df.groupby('Region')['TWh'].transform('sum')` gives each row its region total

In [7]:
# Task 3
fossil_source_order = ['Coal', 'Oil', 'Natural Gas']
all_region_sources = pd.MultiIndex.from_product(
    [df_fossil['Region'].unique(), fossil_source_order],
    names=['Region', 'Source'],
)
df_region_source = (
    df_fossil.groupby(['Region', 'Source'])['TWh']
    .sum()
    .reindex(all_region_sources, fill_value=0)
    .reset_index()
)
df_region_source['RegionTotal'] = df_region_source.groupby('Region')['TWh'].transform('sum')
df_region_source['Share_pct'] = df_region_source['TWh'] / df_region_source['RegionTotal']
df_region_source['Coal_label'] = np.where(
    df_region_source['Source'].eq('Coal'),
    df_region_source['Share_pct'].map(lambda share: f'{share:.0%}'),
    '',
)

coal_by_region = df_region_source.loc[df_region_source['Source'].eq('Coal')]
coal_leader = coal_by_region.loc[coal_by_region['Share_pct'].idxmax()]
region_order = coal_by_region.sort_values('Share_pct')['Region'].tolist()
fossil_mix_colors = {
    'Coal': fossil_colors['Coal'],
    'Oil': '#A6A6A6',
    'Natural Gas': '#D9D9D9',
}

fig_coal_share = px.bar(
    df_region_source,
    x='Share_pct',
    y='Region',
    color='Source',
    text='Coal_label',
    orientation='h',
    color_discrete_map=fossil_mix_colors,
    category_orders={
        'Source': fossil_source_order,
        'Region': region_order,
    },
    labels={'Share_pct': 'Share of regional fossil energy', 'Source': 'Fossil source'},
    title=(f"{coal_leader['Region']} relies most on coal — "
           f"{coal_leader['Share_pct']:.1%} of its fossil energy"),
)
fig_coal_share.update_traces(
    texttemplate='%{text}',
    textposition='inside',
    insidetextanchor='middle',
    textfont=dict(color='white'),
)
fig_coal_share.update_layout(
    template='plotly_white',
    barmode='stack',
    height=500,
    margin=dict(t=100, l=10, r=10, b=10),
    xaxis=dict(title='Share of regional fossil energy', tickformat='.0%', range=[0, 1]),
    yaxis=dict(title='', categoryorder='array', categoryarray=region_order),
    legend=dict(orientation='h', x=0, y=1.05),
)
fig_coal_share.show()

### Task 3 — reflection

**Your Task 1 treemap and this stacked bar show the same fossil data. Name one question each chart answers that the other cannot.**

The treemap answers which countries and fossil sources contribute the most TWh within a region; the stacked bar cannot show country-level detail. The stacked bar answers which region has the highest coal share of its fossil mix; the treemap’s area-focused hierarchy makes that cross-region percentage comparison difficult.